# SUSHRUTA — Baseline-01: FLAIR-Only 2D U-Net Segmentation Baseline

## Executive Summary
This notebook documents the training and held-out evaluation of **Baseline-01**, our lightweight 2D U-Net (~4.3M parameters) trained strictly on 2D FLAIR MRI slices.

### Rigorous Baseline Standards:
- **Data Isolation**: 360 complete locally available patient cases partitioned according to `split_manifest.json` (245 Train, 58 Val, 57 Test).
- **Zero Pre-trained Weights**: Model trained strictly from random initialization (`seed=42`).
- **Loss & Optimizer**: Combined Dice Loss + BCE Loss with AdamW optimizer.
- **Held-Out Evaluation**: Multi-metric evaluation (Global Pixel-Level, Macro Per-Image, and Tumor-Only Slices) conducted exclusively on 57 held-out test patients.
- **Single Modality Benchmark**: Serves as our official reference benchmark for all future multi-modal, lightweight, and real-time acceleration experiments.

---
## Section 1: Environment Setup & Authoritative DataLoaders

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Add project root to sys.path
current_dir = os.path.abspath('')
project_root = current_dir if os.path.exists(os.path.join(current_dir, 'src')) else os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.models.unet import UNet2D, count_parameters
from src.models.dataset import create_brats_dataloaders
from src.models.train_unet import set_seed, UNetTrainer

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dataloaders = create_brats_dataloaders(
    data_dir=os.path.join(project_root, 'data', 'processed', 'brats'),
    batch_size=16,
    num_workers=0,
    seed=42
)

print(f"Train samples: {len(dataloaders['train'].dataset):,}")
print(f"Val samples:   {len(dataloaders['val'].dataset):,}")
print(f"Test samples:  {len(dataloaders['test'].dataset):,}")

Using device: cpu
Train samples: 13,150
Val samples:   3,118
Test samples:  3,076


---
## Section 2: Model Architecture (Lightweight 2D U-Net)

In [ ]:
model = UNet2D(in_channels=1, out_channels=1, base_channels=32).to(device)
n_params = count_parameters(model)
print(f"Model Name: UNet2D")
print(f"Total Trainable Parameters: {n_params:,}")

Model Name: UNet2D
Total Trainable Parameters: 4,317,825


---
## Section 3: Training Loss & Metric Curves

In [ ]:
fig_path = os.path.join(project_root, 'results', 'figures', 'baseline_01_training_curves.png')
if os.path.exists(fig_path):
    img = Image.open(fig_path)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Baseline-01 Training Curves", fontsize=14, fontweight='bold')
    plt.show()

[FIGURE] Displaying training curves from results/figures/baseline_01_training_curves.png


---
## Section 4: Held-Out Test Evaluation & Visual Overlays

In [ ]:
metrics_path = os.path.join(project_root, 'results', 'metrics', 'baseline_01_test_metrics.json')
with open(metrics_path, 'r') as f:
    test_results = json.load(f)

print("=== HELD-OUT TEST METRICS SUMMARY ===")
print(json.dumps(test_results, indent=2))

preds_fig_path = os.path.join(project_root, 'results', 'figures', 'baseline_01_test_predictions.png')
if os.path.exists(preds_fig_path):
    img_preds = Image.open(preds_fig_path)
    plt.figure(figsize=(16, 28))
    plt.imshow(img_preds)
    plt.axis('off')
    plt.title("Baseline-01 Sample Test Predictions Overlay", fontsize=14, fontweight='bold')
    plt.show()

=== HELD-OUT TEST METRICS SUMMARY ===
{
    "experiment_name": "Baseline-01: FLAIR-only 2D U-Net",
    "model_name": "UNet2D",
    "best_epoch": 2,
    "data_summary": {
        "available_patients": 360,
        "train_patients": 245,
        "val_patients": 58,
        "test_patients": 57,
        "train_slices": 13150,
        "val_slices": 3118,
        "test_slices": 3076
    },
    "global_pixel_metrics": {
        "dice": 0.7105531658211027,
        "iou": 0.5510527049171232,
        "precision": 0.5519609993416466,
        "recall": 0.9970226482811232,
        "f1": 0.7105527070989083
    },
    "macro_per_image_all_slices": {
        "dice": 0.7990068183466947,
        "iou": 0.74737100154163,
        "precision": 0.7487858882174404,
        "recall": 0.9010378178776517,
        "f1": 0.8789656749740243
    },
    "tumor_only_slice_metrics": {
        "dice": 0.8069781440117707,
        "iou": 0.7350118488130742,
        "precision": 0.7369838163054929,
        "recall": 0.949